In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Cookbook_Form_Teleportation)=
# Form Conversions

*Converting structures across OpenMM, MDAnalysis, NetworkX, MDTraj, and BioPython.*

No single scientific library solves every structural biology problem. Specialized tools exist for graph theory (NetworkX), trajectory mechanics (OpenMM, MDAnalysis, MDTraj), and sequence alignment (BioPython). MolSysMT acts as a universal bridge across these ecosystems, enabling instant zero-loss conversions.

In this recipe, we convert a single molecular system across 5 different Python libraries and verify conversion fidelity using MolSysMT's conversion auditing engine.

:::{versionadded} 1.0.0
:::


## Loading System

We load the Trp-Cage miniprotein as our starting molecular system:

In [2]:
import molsysmt as msm

# Load base system
molsys = msm.convert(msm.systems['Trp-Cage']['1l2y.pdb'], to_form='molsysmt.MolSys', structure_indices=[0])
msm.info(molsys)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,304,20,1,1,1,1,1,1


## OpenMM Conversion

We convert to an `openmm.Topology` with an explicit conversion audit report:

In [3]:
# Convert to OpenMM Topology with audit report
openmm_top, report = msm.convert(molsys, to_form='openmm.Topology', return_report=True)
print(f"OpenMM Topology created with {openmm_top.getNumAtoms()} atoms.")
print(f"Conversion outcome: {report.outcome} (exhaustive audit: {report.is_exhaustive})")

OpenMM Topology created with 304 atoms.
Conversion outcome: lossy (exhaustive audit: False)


## MDAnalysis Conversion

We convert the raw PDB into an `MDAnalysis.Universe` and then back to `molsysmt.MolSys`:

In [4]:
# Convert PDB to MDAnalysis Universe
mda_universe = msm.convert(msm.systems['Trp-Cage']['1l2y.pdb'], to_form='MDAnalysis.Universe')
print(f"MDAnalysis Universe created with {len(mda_universe.atoms)} atoms and {len(mda_universe.residues)} residues.")

MDAnalysis Universe created with 304 atoms and 20 residues.


## NetworkX Conversion

We convert the covalent topology into a `networkx.Graph` to analyze molecular network connectivity:

In [5]:
import networkx as nx

# Convert covalent bonds to NetworkX Graph
graph = msm.convert(molsys, to_form='networkx.Graph')
print(f"NetworkX Graph: {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
print(f"Graph is connected: {nx.is_connected(graph)}")

NetworkX Graph: 304 nodes and 310 edges.
Graph is connected: True


## BioPython Conversion

We extract the sequence directly into a `biopython.Seq` object:

In [6]:
# Convert to BioPython Sequence
biopython_seq = msm.convert(molsys, to_form='biopython.Seq')
print(f"BioPython Amino Acid Sequence: {biopython_seq}")

BioPython Amino Acid Sequence: NLYIQWLKDGGPSSGRPPPS


## MDTraj Conversion

We convert the system to an `mdtraj.Trajectory` object:

In [7]:
# Convert to MDTraj Trajectory
mdtraj_traj = msm.convert(molsys, to_form='mdtraj.Trajectory')
print(f"MDTraj Trajectory: {mdtraj_traj.n_frames} frame, {mdtraj_traj.n_atoms} atoms.")

MDTraj Trajectory: 1 frame, 304 atoms.


## Round Trip Restoration

Any external object can be converted back to `molsysmt.MolSys` without loss of topological invariants:

In [8]:
# Convert from MDAnalysis Universe back to MolSysMT
molsys_restored = msm.convert(mda_universe, to_form='molsysmt.MolSys')
msm.info(molsys_restored)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_peptides,n_structures
molsysmt.MolSys,304,20,304,1,1,1,1,38


:::{seealso}
:class: dropdown

- {ref}`Introduction_Forms`: Comprehensive catalog of supported molecular system forms.
- {func}`molsysmt.basic.convert`: Universal conversion gateway between all forms.
- {ref}`Tutorial_Form_openmm_Topology`: OpenMM Topology form adapter.
- {ref}`Tutorial_Form_MDAnalysis_Universe`: MDAnalysis Universe form adapter.
:::